# London Clustering, Tucker Tensor Decomposition

Exploratory notebook, first pass, not part of any book chapter. Building
the London-side counterpart to the GoiEner Tucker notebook, one of four
alternative representations Part 5 Chapter 1 promises get checked against
the settled recipe "including households in London and Spain." Only the
GoiEner side existed until now.

A Tucker decomposition respects a household's own 3-way structure,
households x days x half-hours, directly: it finds a small number of
shared temporal factors common to every real household, plus each
household's own loading onto those factors. That per-household loading is
the low-dimensional embedding clustered on below, built by construction
rather than by hoping a scaler and a lot of columns sort themselves out.

## Getting the data

The same shared loader Chapter 2 and Chapter 3 already use for London,
1,284 real households after coverage filtering, half-hourly, not hourly
like GoiEner.

In [ ]:
from lets_plot import LetsPlot
import numpy as np
import pandas as pd

from ark.cluster.datasets import load_london_pivot
from ark.cluster.preprocessing import map_to_seasons

LetsPlot.setup_html(isolated_frame=True)
N_PARTITIONS = 50
WINDOW_START = "2013-01-01"
WINDOW_DAYS = 360
MIN_COVERAGE = 0.999
STEPS_PER_DAY = 48  # real, disclosed mismatch: London is half-hourly, not hourly like GoiEner

pivot = load_london_pivot(
    n_partitions=N_PARTITIONS,
    window_start=WINDOW_START,
    window_days=WINDOW_DAYS,
    min_coverage=MIN_COVERAGE,
)
n_customers = pivot.shape[1]
household_ids = list(pivot.columns)
print(
    f"pivot: {pivot.shape} (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader"
)

pivot: (17280, 1284) (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader


In [ ]:
def season_tensor(pivot: pd.DataFrame, dates: pd.DatetimeIndex) -> np.ndarray:
    """Real (households, days, half-hours) tensor for a real date subset, households in pivot's own column order."""
    subset = pivot[pivot.index.normalize().isin(dates)]
    return subset.T.to_numpy().reshape(subset.shape[1], len(dates), STEPS_PER_DAY)


day_index = pd.date_range(WINDOW_START, periods=WINDOW_DAYS, freq="D")
# Real calendar summer (Northern Hemisphere), matching the GoiEner Tucker
# notebook's own convention and Chapter 2/3's own settled recipe.
summer_dates = day_index[map_to_seasons(day_index, hemisphere="north") == "summer"]
season = season_tensor(pivot, summer_dates)
print(f"season tensor: {season.shape} (customers, real summer days, half-hours)")

season tensor: (1284, 92, 48) (customers, real summer days, half-hours)


## Tucker decomposition: a genuinely 3-way structure, not a flattened one

The season tensor above is already a real 3-way array, households x days
x half-hours. Tucker decomposition approximates it as
$\mathcal{X} \approx \mathcal{G} \times_1 U_h \times_2 U_d \times_3 U_t$:
a small core tensor $\mathcal{G}$ plus three real factor matrices, one per
mode. $U_h$, the household-mode factor matrix, is exactly the
low-dimensional embedding this notebook clusters on.

In [ ]:
import tensorly as tl
from tensorly.decomposition import tucker

tl.set_backend("numpy")
tensor = tl.tensor(season)

# Day-mode and half-hour-mode ranks fixed at a modest, real size, the same
# convention the GoiEner Tucker notebook uses; household-mode rank is what
# this notebook actually varies and reports, since that is the real
# embedding width the downstream clustering step depends on.
DAY_RANK = 10
HALF_HOUR_RANK = 8
household_rank_candidates = [5, 10, 15, 20, 30]

reconstruction_rows = []
for r_h in household_rank_candidates:
    core, factors = tucker(tensor, rank=[r_h, DAY_RANK, HALF_HOUR_RANK], random_state=0)
    reconstructed = tl.tucker_to_tensor((core, factors))
    real_variance = float(np.var(season))
    residual_variance = float(np.var(season - reconstructed))
    explained = 1.0 - residual_variance / real_variance
    reconstruction_rows.append({"Household rank": r_h, "Explained variance": explained})

reconstruction_df = pd.DataFrame(reconstruction_rows)

from ark.plot.gt_style import themed_gt

themed_gt(reconstruction_df.round(4))

Household rank,Explained variance
5,0.4407
10,0.4771
15,0.4955
20,0.504
30,0.5143


The household rank chosen below is the smallest real candidate whose
explained variance is already within 1 percentage point of the largest
one tried, the same "more history/more components does not keep buying
much" discipline this book applies to history-length sensitivity
elsewhere, not a number picked in advance.

In [ ]:
best_explained = reconstruction_df["Explained variance"].max()
HOUSEHOLD_RANK = int(
    reconstruction_df.loc[reconstruction_df["Explained variance"] >= best_explained - 0.01, "Household rank"].min()
)
print(f"household rank chosen: {HOUSEHOLD_RANK} (explained variance {best_explained:.4f} at the largest rank tried)")

core, factors = tucker(tensor, rank=[HOUSEHOLD_RANK, DAY_RANK, HALF_HOUR_RANK], random_state=0)
U_household = factors[0]
print(f"U_household: {U_household.shape} (customers, household-mode rank), the real per-household embedding")

household rank chosen: 30 (explained variance 0.5143 at the largest rank tried)


U_household: (1284, 30) (customers, household-mode rank), the real per-household embedding


## Clustering on the Tucker household embedding

Same validity-curve procedure as the GoiEner Tucker notebook, applied to
this genuinely different, by-construction low-dimensional
representation.

In [ ]:
from ark.cluster.cluster_validation import clustering_validity_scores
from ark.plot.clustering import validity_curve

scores_tucker = clustering_validity_scores(U_household, range(2, 10))
themed_gt(scores_tucker.round(3))

k,inertia,bic,silhouette,calinski_harabasz,davies_bouldin
2,28.375,,0.813,40.167,0.124
3,27.762,,0.777,34.649,0.14
4,26.95,,0.264,36.627,2.661
5,26.432,,0.152,34.257,3.085
6,25.818,,0.335,34.112,2.497
7,25.323,,0.257,33.118,2.215
8,24.92,,0.191,31.772,2.506
9,24.484,,0.24,31.117,2.329


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_tucker)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_TUCKER = int(scores_tucker.loc[scores_tucker["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER}")

from sklearn.cluster import KMeans

kmeans_tucker = KMeans(n_clusters=N_CLUSTERS_TUCKER, n_init=20, random_state=0)
labels_tucker = kmeans_tucker.fit_predict(U_household)
table_tucker = pd.DataFrame({"labels": labels_tucker}).value_counts().sort_index().reset_index()
table_tucker.columns = ["Label", "Count"]
themed_gt(table_tucker)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,37
1,1247


Raw Tucker decomposition does not remove each real household's own scale
before decomposing, unlike the book's own shape-only convention. Checked
directly below: does peak-normalizing each real household's own season
first change what this genuinely different methodology finds, the same
question the GoiEner Tucker notebook already asked and answered there.

In [ ]:
household_peak = season.max(axis=(1, 2), keepdims=True)
household_peak = np.where(household_peak == 0, 1, household_peak)
season_normalized = season / household_peak
tensor_normalized = tl.tensor(season_normalized)

core_norm, factors_norm = tucker(tensor_normalized, rank=[HOUSEHOLD_RANK, DAY_RANK, HALF_HOUR_RANK], random_state=0)
U_household_norm = factors_norm[0]

scores_tucker_norm = clustering_validity_scores(U_household_norm, range(2, 10))
themed_gt(scores_tucker_norm.round(3))

k,inertia,bic,silhouette,calinski_harabasz,davies_bouldin
2,28.354,,0.314,36.586,2.008
3,27.716,,0.063,33.455,4.25
4,27.199,,0.036,30.815,4.313
5,26.681,,0.029,29.745,3.559
6,26.233,,0.023,28.554,3.413
7,25.83,,0.016,27.464,3.369
8,25.402,,0.009,26.99,3.153
9,25.112,,0.0,25.715,3.207


In [ ]:
N_CLUSTERS_TUCKER_NORM = int(scores_tucker_norm.loc[scores_tucker_norm["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER_NORM}")

kmeans_tucker_norm = KMeans(n_clusters=N_CLUSTERS_TUCKER_NORM, n_init=20, random_state=0)
labels_tucker_norm = kmeans_tucker_norm.fit_predict(U_household_norm)
table_tucker_norm = pd.DataFrame({"labels": labels_tucker_norm}).value_counts().sort_index().reset_index()
table_tucker_norm.columns = ["Label", "Count"]
themed_gt(table_tucker_norm)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,1246
1,38


## Is either split actually trustworthy?

Both silhouette curves above are exactly the kind of signal Part 5
Chapter 2 built its trust-gated composite score to interrogate rather
than trust at face value. The same real audit already run on every
GoiEner notebook, prediction strength and cluster-wise bootstrap
stability, applied here for the first time to a Tucker-decomposed
London embedding.

In [ ]:
from scipy.stats import entropy
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

from ark.cluster.cluster_validation import composite_trustworthy_score
from ark.cluster.stability import cluster_stability, prediction_strength


def _kmeans_cluster_fn(X_arr, k):
    return KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_arr)


def trust_gate_audit(X_embedding, k_range):
    rows = []
    for k in k_range:
        labels_k = _kmeans_cluster_fn(X_embedding, k)
        sizes = np.bincount(labels_k)
        ps = prediction_strength(X_embedding, k_range=[k], cluster_fn=_kmeans_cluster_fn, n_splits=10, random_state=0)[
            "prediction_strength"
        ].iloc[0]
        stability_scores = cluster_stability(
            X_embedding, labels_k, cluster_fn=_kmeans_cluster_fn, n_bootstrap=100, random_state=0
        )
        rows.append(
            {
                "k": k,
                "silhouette": silhouette_score(X_embedding, labels_k),
                "calinski_harabasz": calinski_harabasz_score(X_embedding, labels_k),
                "davies_bouldin": davies_bouldin_score(X_embedding, labels_k),
                "balance": entropy(sizes) / np.log(k),
                "prediction_strength": ps,
                "min_cluster_stability": min(stability_scores.values()),
            }
        )
    return pd.DataFrame(rows)


trust_audit_raw_df = trust_gate_audit(U_household, [2])
print("--- raw Tucker embedding ---")
themed_gt(trust_audit_raw_df.round(3))

--- raw Tucker embedding ---


k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.455,39.818,2.184,0.188,0.96,0.177


In [ ]:
trust_audit_norm_df = trust_gate_audit(U_household_norm, range(2, 6))
print("--- peak-normalized Tucker embedding ---")
themed_gt(trust_audit_norm_df.round(3))

--- peak-normalized Tucker embedding ---


k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.291,37.063,2.082,0.192,0.812,0.384
3,0.028,33.001,4.546,0.716,0.515,0.411
4,0.02,30.605,4.315,0.863,0.438,0.274
5,0.02,29.474,3.953,0.793,0.36,0.157


In [ ]:
trust_gated_norm_df = composite_trustworthy_score(
    trust_audit_norm_df.rename(columns={"k": "trial_number"}),
    trust_metrics=("balance", "min_cluster_stability"),
    id_column="trial_number",
).rename(columns={"trial_number": "k"})
themed_gt(trust_gated_norm_df.round(3))

k,separation_score,trust_factor,composite_score
3,0.556,0.411,0.229
2,1.0,0.192,0.192
4,0.427,0.274,0.117
5,0.516,0.157,0.081


## Summary

A genuinely different, harder result than the GoiEner Tucker notebook's
own findings, worth reporting exactly as found.

**The Tucker decomposition itself compresses London far less cleanly than
GoiEner.** GoiEner's own 2,000-household season tensor reached 99.25%
explained variance at household rank 5; London's 1,284-household tensor
reaches only 51.43% even at rank 30, the largest candidate tried, with
explained variance still rising, not plateauing, at that point. London's
half-hourly households (48 steps a day against GoiEner's 24) carry real,
finer-grained behavioural variation a low-rank household factor struggles
to summarise. The household rank chosen here, 30, is therefore a real,
disclosed compromise, not a settled answer the way GoiEner's rank-5 choice
was.

**Raw Tucker reproduces the same magnitude-dominated warning sign already
flagged in GoiEner.** Its own validity curve is suspiciously high at
$k{=}2$ and $k{=}3$ (0.813, 0.777) before collapsing sharply at $k{=}4$
(0.264), the same shape that earlier flagged an outlier-isolating,
magnitude-driven embedding rather than genuine structure. The chosen
$k{=}2$ splits a real 37 households against 1,247, a tight 2.9% minority.

**Peak-normalizing does not resolve the imbalance, it just makes the
curve honest.** The peak-normalized validity curve falls monotonically
from 0.314 at $k{=}2$ to 0.000 at $k{=}9$, a well-behaved shape with no
suspicious jump, but the chosen $k{=}2$ still splits a near-identical 38
households against 1,246. Normalizing away each household's own scale
removed the misleading curve shape without removing the lopsided split
underneath it.

**Neither embedding survives the real trust gate on this population.**
Raw Tucker's own $k{=}2$ posts an excellent prediction strength, 0.960,
but fails cluster-wise stability outright, 0.177, deep in Hennig's
dissolved band: exactly the kind of split prediction strength alone would
wrongly certify, the precise failure mode the composite score exists to
catch. Peak-normalized Tucker fares no better across every $k$ checked,
2 through 5: prediction strength never clears 0.812, and every value sits
well under Tibshirani and Walther's 0.8 floor from $k{=}3$ onward (0.515,
0.438, 0.360), while stability never exceeds 0.411. The composite ranking
places $k{=}3$ marginally ahead of $k{=}2$ (0.229 against 0.192), but
that ordering is close to meaningless here: neither clears the real
resampling floor this book applies everywhere else. Unlike the GoiEner
Tucker notebook, which found $k{=}2$ and $k{=}3$ both landing in a
genuine, close landscape of validated candidates, London's Tucker
embedding does not clear that bar at all. That is itself the honest
finding: on London, a Tucker household factorisation, at the ranks and
$k$ values checked here, does not produce a trustworthy alternative to
the settled `shape`+PaCMAP+GMM recipe's own rock-solid $k{=}2$
(prediction strength 1.000, stability 1.000).